First we will download some of the data from Gaia DR3, in this case we will be downloading the first 2 csv files in the source directory.

In [ ]:
import os
import requests
import gzip
import shutil
import pandas as pd
import glob
from astropy.coordinates import SkyCoord, search_around_sky
from astropy import units as u
from bs4 import BeautifulSoup
from urllib.parse import urljoin

In [ ]:
# Total size ~500 MB
url = "http://cdn.gea.esac.esa.int/Gaia/gdr3/gaia_source/"
output_folder = "GaiaDR3_Source"

# Create output directory
os.makedirs(output_folder, exist_ok=True)

response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

N = 2 # Number of files we want to read

# Find all links ending in .csv.gz
all_files = [a['href'] for a in soup.find_all('a') if a['href'].endswith('.csv.gz')]
files = all_files[:N]

print(f"Found {len(all_files)} total files. Processing the first {len(files)}...")

for filename in files:
    file_url = urljoin(url, filename)
    local_gz_path = os.path.join(output_folder, filename)
    local_csv_path = os.path.join(output_folder, filename[:-3]) #removes the .gz extension

    # Check if csv is already downloaded
    if os.path.exists(local_csv_path):
        print(f"Skipping {filename} (unzipped file already exists)")
        continue

    # Check if compressed file needs to be downloaded
    if not os.path.exists(local_gz_path):
        print(f"Downloading {filename}...")
        try:
            with requests.get(file_url, stream=True) as r:
                r.raise_for_status()
                with open(local_gz_path, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=32768):
                        f.write(chunk)
        except Exception as e:
            print(f"Failed to download {filename}: {e}")
            if os.path.exists(local_gz_path):
                os.remove(local_gz_path)
            continue
    else:
        print(f"Skipping download for {filename} (compressed file exists)")

    # Uncompress downloaded file for use
    print(f"Unzipping {filename}...")
    try:
        with gzip.open(local_gz_path, 'rb') as f_in:
            with open(local_csv_path, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        # Delete the .gz file after unzipping to save space
        os.remove(local_gz_path)
    except Exception as e:
        print(f"Failed to unzip {filename}: {e}")
        # Cleanup corrupted output if unzip fails
        if os.path.exists(local_csv_path):
            os.remove(local_csv_path)

print("Download complete.")

Now we will cross match the data with the TESS data set from sector 1.

In [ ]:
def cross_match(gaia_folder, tess_csv_path, threshold_degree, output_filename, output_dir):
    '''
    :param gaia_folder: Directory containing the downloaded Gaia .csv or .csv.gz files
    :param tess_csv_path: Full path to the single TESS CSV file
    :param threshold_degree: Match radius in degrees
    :param output_filename: Filename of the result csv file
    :param output_dir: Directory to save the result
    '''

    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, output_filename)


    if os.path.exists(output_path):
        print(f"Skipping {output_filename} (already exists)")
        return

    # Load the Gaia Data
    print(f"Loading Gaia data from: {gaia_folder}")

    # Find all CSV or CSV.GZ files
    gaia_files = glob.glob(os.path.join(gaia_folder, "*.csv*"))

    if not gaia_files:
        print("No CSV files found in Gaia directory!")
        return

    gaia_dfs = []
    for file in gaia_files:
        try:
            # Pandas handles .csv.gz automatically if extension is present
            df = pd.read_csv(file, comment='#')
            gaia_dfs.append(df)
        except Exception as e:
            print(f"Error reading {file}: {e}")

    if not gaia_dfs:
        print("Could not load any Gaia data.")
        return

    # Combine all individual files into one big table
    gaia_catalog = pd.concat(gaia_dfs, ignore_index=True)
    print(f"Total Gaia stars loaded: {len(gaia_catalog)}")

    # Ensure column names match Gaia DR3 (usually lower case)
    gaia_ra_col = 'ra'
    gaia_dec_col = 'dec'
    gaia_id_col = 'source_id'

    # Load Tess Data
    print(f"Loading TESS data from: {tess_csv_path}")
    tess_catalog = pd.read_csv(tess_csv_path, sep=r'\s+')

    # DEFINE TESS COLUMNS HERE
    tess_ra_col = 'ra'
    tess_dec_col = 'dec'
    tess_id_col = 'name'

    # Check if columns exist
    if tess_ra_col not in tess_catalog.columns:
        print(f"Error: Column '{tess_ra_col}' not found in TESS CSV. Please update the script variables.")
        return

    print(f"Total TESS stars loaded: {len(tess_catalog)}")

    # Perform cross-match
    print(f"Matching with threshold: {threshold_degree} deg...")

    # Create coordinate objects
    c_gaia = SkyCoord(ra=gaia_catalog[gaia_ra_col].values * u.degree,
                      dec=gaia_catalog[gaia_dec_col].values * u.degree)

    c_tess = SkyCoord(ra=tess_catalog[tess_ra_col].values * u.degree,
                      dec=tess_catalog[tess_dec_col].values * u.degree)

    # search_around_sky finds all matches within the limit
    # idx_tess: indices of tess stars that matched
    # idx_gaia: indices of gaia stars that matched
    idx_tess, idx_gaia, d2d, d3d = search_around_sky(c_tess, c_gaia, threshold_degree * u.degree)

    print(f"Found {len(idx_tess)} matches.")

    # Build .csv output
    # Extract matched rows
    matches_tess = tess_catalog.iloc[idx_tess].reset_index(drop=True)
    matches_gaia = gaia_catalog.iloc[idx_gaia].reset_index(drop=True)

    # Create final DataFrame combining both info
    result_df = pd.DataFrame({
        'TESS_ID': matches_tess[tess_id_col],
        'TESS_RA': matches_tess[tess_ra_col],
        'TESS_Dec': matches_tess[tess_dec_col],
        'Gaia_ID': matches_gaia[gaia_id_col],
        'Gaia_RA': matches_gaia[gaia_ra_col],
        'Gaia_Dec': matches_gaia[gaia_dec_col],
        'Separation_Deg': d2d.value
    })

    # Add extra columns if available (e.g., classification)
    if 'best_class_name' in matches_gaia.columns:
        result_df['Gaia_Class'] = matches_gaia['best_class_name']
    elif 'ML_classification' in matches_gaia.columns:
        result_df['Gaia_Class'] = matches_gaia['ML_classification']

    # Save the file to an output directory
    result_df.to_csv(output_path, index=False)
    print(f"Saved match results to: {output_path}")

Run the cross_match function.

In [ ]:
TessObjects_folder = "TessObjects"
os.makedirs(TessObjects_folder, exist_ok=True)
TessObjects_file_name = "count_transients.csv"
TessObjects_path = os.path.join(TessObjects_folder, TessObjects_file_name)

out_file_name = "GaiaDR3_Source_TessObjects.csv"
out_file_folder = "GaiaDR3_Cross_Match_Data"

cross_match(output_folder, TessObjects_path, 0.01, out_file_name, out_file_folder)